In [1]:
from pathlib import Path
import os 
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog
import unicodedata
from rapidfuzz import process
from clase_reportes_new import ReportClassNew
from send_mail import MailSender
rc = ReportClassNew()
ms = MailSender()


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os 

ruta = Path(r'G:/Otros ordenadores/Mi portátil/VENTA MENSUAL')
ruta_base = r'g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\contabilidad\base_estado_resultados.xlsx'

base = pd.read_excel(ruta_base)
# Definición de rutas específicas para cada tipo de balance (pcn comentado por el momento)
ruta_pcn = ruta / 'data' / 'contabilidad' / 'balance_comprobacion' / 'pcn'
ruta_hfa = ruta / 'data' / 'contabilidad' / 'balance_comprobacion' / 'hfa'

# Lista de carpetas a iterar para la lectura de archivos
rutas = [
    ruta_hfa,
    ruta_pcn
]

# Diccionario de mapeo para convertir abreviaturas de meses en números de mes
meses = {
    'ene': 1, 'feb': 2, 'mar': 3, 'abr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'ago': 8, 'sept': 9, 'oct': 10, 'nov': 11, 'dic': 12
}

# Lista para almacenar los DataFrames procesados individualmente
dfs = []

# Ciclo principal para recorrer cada carpeta de empresa definida en 'rutas'
for ruta_carpeta in rutas:
    try:    
        # Listar archivos dentro de la carpeta actual
        for archivo in os.listdir(ruta_carpeta):
            # Filtrar solo archivos con extensión .xlsx
            if archivo.endswith(f'.{"xlsx"}'):
                ruta_completa = os.path.join(ruta_carpeta, archivo)
                print(ruta_completa)
                
                # Carga del archivo Excel
                df_balance= pd.read_excel(ruta_completa)
                print(ruta_completa)
                
                # Extracción del periodo (mes y año) desde la quinta columna del encabezado original
                valor = df_balance.columns[4]  # ej: 'ene 2026'
                mes_txt, anio = valor.split()
                # Creación de un objeto Timestamp para representar el primer día de ese mes/año
                fecha_informe = pd.Timestamp(int(anio), meses[mes_txt.lower()], 1)
                print('jhhhj')
                
                # Reasignación de encabezados usando la segunda fila del Excel (índice 1)
                df_balance.columns = df_balance.iloc[1].values
                
                # Definición manual de nombres de columnas para asegurar consistencia en el análisis
                cols = list(df_balance.columns)
                cols[0] = 'cuenta'
                cols[1] = 'descripcion'
                cols[2] = 'debe_inicial'
                cols[3] = 'haber_inicial'

                cols[4] = 'debe'
                cols[5] = 'haber'

                cols[6] = 'debe_final'
                cols[7] = 'haber_final'

                df_balance.columns = cols
                print(df_balance.columns)
                
                # Eliminación de filas que no contienen un número de cuenta (limpieza de basura del Excel)
                df_balance = df_balance[df_balance['cuenta'].notna()]

                # Identificación de la empresa propietaria de los datos según el nombre de la carpeta
                if ruta_carpeta.name == 'hfa':
                    df_balance['empresa'] = 'hfa'
                elif ruta_carpeta.name == 'pcn':
                    df_balance['empresa'] = 'pcn'
                else:
                    df_balance['empresa'] = 'desconocido'

                # Asignación de la fecha de informe extraída previamente
                df_balance['fecha_informe'] = fecha_informe

                # Conversión de la columna 'cuenta' a entero para facilitar filtrado y segmentación
                df_balance['cuenta'] = df_balance['cuenta'].astype(int)

                # Agregar el DataFrame procesado a la lista colectora
                dfs.append(df_balance)

    except Exception as e:
        # Manejo de errores durante el procesamiento de archivos por carpeta
        print(f"Error al procesar la info en {ruta_carpeta}: {e}")

# Consolidación de todos los DataFrames individuales en uno solo
df_consolidado = pd.concat(dfs, ignore_index=True, sort=False)

# Generación de niveles de jerarquía contable basados en la longitud del código de cuenta
df_consolidado['nivel_1'] = df_consolidado['cuenta'].astype(str).str[:1]
df_consolidado['nivel_2'] = df_consolidado['cuenta'].astype(str).str[:2]
df_consolidado['nivel_3'] = df_consolidado['cuenta'].astype(str).str[:4]
df_consolidado['nivel_4'] = df_consolidado['cuenta'].astype(str).str[:6]
df_consolidado['nivel_5'] = df_consolidado['cuenta'].astype(str).str[:8]

# Mapeo de la naturaleza contable (Deudora/Acreedora) según el primer dígito (Clase)
map_naturaleza = {
    '1': 'deudora',   # Activo
    '5': 'deudora',   # Gastos
    '6': 'deudora',   # Costos
    '2': 'acreedora', # Pasivo
    '3': 'acreedora', # Patrimonio
    '4': 'acreedora'  # Ingresos
}

cols = [
    'debe_inicial', 'haber_inicial',
    'debe', 'haber',
    'debe_final', 'haber_final',
]
df_consolidado[cols] = df_consolidado[cols].astype(float)
df_consolidado['naturaleza'] = df_consolidado['nivel_1'].map(map_naturaleza)

# Cálculo del Saldo Inicial neto ajustado según la naturaleza de la cuenta
df_consolidado['saldo_inicial'] = np.where(
    df_consolidado['naturaleza'] == 'deudora',
    df_consolidado['debe_inicial'] - df_consolidado['haber_inicial'],
    df_consolidado['haber_inicial'] - df_consolidado['debe_inicial']
    
)

# Cálculo del Saldo Final neto ajustado según la naturaleza de la cuenta
df_consolidado['movimiento_neto'] = np.where(
    df_consolidado['naturaleza'] == 'deudora',
    df_consolidado['debe_final'] - df_consolidado['haber_final'],
    df_consolidado['haber_final'] - df_consolidado['debe_final']


)

# 2. El Saldo Final REAL es la suma del inicial más el movimiento
df_consolidado['saldo_final'] = df_consolidado['saldo_inicial'] + df_consolidado['movimiento_neto']

# Verificación matemática del saldo final basado en movimientos del periodo
df_consolidado['saldo_calculated'] = df_consolidado['saldo_inicial'] + (
    df_consolidado['debe'] - df_consolidado['haber']
)


# Cálculo del Saldo Final neto ajustado según la naturaleza de la cuenta
df_consolidado['Valor'] = np.where(
    df_consolidado['naturaleza'] == 'deudora',
    df_consolidado['debe'] - df_consolidado['haber'],
    df_consolidado['haber'] - df_consolidado['debe']


)


# --- Asignación de la columna 'Valor' para reportes financieros específicos ---

# # Para Activos (1), el valor es igual al saldo final
# df_consolidado.loc[df_consolidado['nivel_1'] == '1', 'Valor'] = df_consolidado['saldo_final']

# # Para Pasivo y Patrimonio (2, 3), el valor se invierte para análisis de Balance General
# df_consolidado.loc[
#     df_consolidado['nivel_1'].isin(['2', '3']),
#     'Valor'
# ] = df_consolidado['saldo_final'] * -1

# # Para Ingresos (4), se toma la diferencia neta del movimiento mensual (Haber - Debe)
# df_consolidado.loc[
#     df_consolidado['nivel_1'] == '4',
#     'Valor'
# ] = df_consolidado['haber'] - df_consolidado['debe']

# # Para Gastos y Costos (5, 6, 7), se toma la diferencia neta del movimiento mensual (Debe - Haber)
# df_consolidado.loc[
#     df_consolidado['nivel_1'].isin(['5', '6', '7']),
#     'Valor'
# ] = df_consolidado['debe'] - df_consolidado['haber']



df_consolidado['nivel_3'] = df_consolidado['nivel_3'].astype(int)
base['N4'] = base['N4'].astype(int)
base_cuenta = base[base['cruce']!='nivel']
base_nivel = base[base['cruce']=='nivel']
base_nivel = base_nivel.drop_duplicates(subset=['N4'])
# Crear el diccionario: {CUENTA: CONCEPTO}
base_cuenta_dicc = base_cuenta.set_index('CUENTA')['CONCEPTO'].to_dict()


df_consolidado = df_consolidado.merge(base_nivel, left_on='nivel_3', right_on='N4', how='left')
# Filtramos las filas donde la cuenta existe en el diccionario
mask = df_consolidado['cuenta'].isin(base_cuenta_dicc.keys())

# Asignamos el mapeo solo a esas filas específicas
df_consolidado.loc[mask, 'CONCEPTO'] = df_consolidado.loc[mask, 'cuenta'].map(base_cuenta_dicc)
cuentas_revision = df_consolidado[df_consolidado['CONCEPTO'].isna()]


G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\contabilidad\balance_comprobacion\hfa\balance_de_comprobación (10).xlsx
G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\contabilidad\balance_comprobacion\hfa\balance_de_comprobación (10).xlsx
jhhhj
Index(['cuenta', 'descripcion', 'debe_inicial', 'haber_inicial', 'debe',
       'haber', 'debe_final', 'haber_final'],
      dtype='object')
G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\contabilidad\balance_comprobacion\hfa\balance_de_comprobación (9).xlsx
G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\contabilidad\balance_comprobacion\hfa\balance_de_comprobación (9).xlsx
jhhhj
Index(['cuenta', 'descripcion', 'debe_inicial', 'haber_inicial', 'debe',
       'haber', 'debe_final', 'haber_final'],
      dtype='object')
G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\contabilidad\balance_comprobacion\hfa\balance_de_comprobación (8).xlsx
G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\contabilidad\balance_comprobacion\hfa

In [2]:
cuentas_revision.to_excel(r'G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\cuentas_revision.xlsx', index=False)